# Auditoría de Robustez y Pruebas Fuera de Distribución (OOD) — Rúbrica B6

Este notebook evalúa sistemáticamente la robustez del sistema frente a **72 imágenes fuera de distribución (OOD)** organizadas en 6 categorías de perturbación (`data/ood/metadata.csv`), comparándolas contra los conjuntos en distribución (`val` y `test_unseen`).

Se analizan **dos escenarios complementarios**:
1. **Juez directo (`y` vs. OOD):** mide la calibración y capacidad de rechazo intrínseca de los clasificadores congelados (`ResNet-18` y `EfficientNet-B0`).
2. **Pipeline completo de despliegue (`G(x)` → Juez):** replica exactamente el flujo de `app/demo_realtime.py`, pasando tanto entradas limpias como entradas OOD por el generador `UNetGenerator` antes de evaluar confianza con el juez.

> **Reutilización para Fase 5:** Para re-ejecutar esta auditoría con el modelo final *fine-tuned* sobre trazos reales, basta con cambiar `DATA_DIR = ROOT_DIR / 'data/real_pairs'` y `GEN_CKPT = ROOT_DIR / 'checkpoints/pix2pix_finetuned.pth'` en la celda de configuración.

In [1]:
from __future__ import annotations

import csv
import json
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve()
if not (ROOT_DIR / 'src').exists():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader

from eval.evaluate_models import load_generator, load_judge
from src.config import IDX_TO_CHAR, IMG_SIZE, SEED
from src.data.dataset import GlyphPairDataset, _pil_to_normalized_tensor
from src.utils import denormalize_tensor, seed_everything

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Parámetros configurables (cambiar a real_pairs y pix2pix_finetuned.pth en Fase 5)
DATA_DIR = ROOT_DIR / 'data' / 'synthetic_pairs'
OOD_DIR = ROOT_DIR / 'data' / 'ood'
GEN_CKPT = ROOT_DIR / 'checkpoints' / 'pix2pix_pretrained.pth'
JUDGES_DIR = ROOT_DIR / 'checkpoints'
FIG_DIR = ROOT_DIR / 'docs' / 'figuras'
RES_DIR = ROOT_DIR / 'results' / 'eval'
DEFAULT_TAU = 0.70

FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

gen = load_generator(GEN_CKPT, device)
judges = {
    'resnet18': load_judge('resnet18', JUDGES_DIR / 'judge_resnet18.pth', device),
    'efficientnet_b0': load_judge('efficientnet_b0', JUDGES_DIR / 'judge_efficientnet_b0.pth', device),
}
print(f'Dispositivo: {device} | Dataset ID: {DATA_DIR.name} | Generador: {GEN_CKPT.name}')

Dispositivo: cpu | Dataset ID: synthetic_pairs | Generador: pix2pix_pretrained.pth


In [2]:
# Cargar catálogo completo de imágenes OOD (>= 50 obligatorias por Rúbrica B6)
ood_df = pd.read_csv(OOD_DIR / 'metadata.csv')
assert len(ood_df) >= 50, f'Se requieren >=50 imágenes OOD, encontradas: {len(ood_df)}'
print(f'Total imágenes OOD: {len(ood_df)}')
display(ood_df['category'].value_counts().rename_axis('Categoría OOD').to_frame('Muestras'))

ood_tensors = []
for rel_path in ood_df['image_path']:
    img = Image.open(OOD_DIR / rel_path).convert('L')
    ood_tensors.append(_pil_to_normalized_tensor(img, IMG_SIZE))
ood_batch = torch.stack(ood_tensors, dim=0)  # [N_ood, 1, 128, 128]
print('Tensor OOD:', tuple(ood_batch.shape))

Total imágenes OOD: 72


,Muestras
Categoría OOD,
ruido_puro,12
garabatos,12
simbolos_ineditos,12
rotacion_extrema,12
desenfoque_corte,12
sombra_bajo_contraste,12


Tensor OOD: (72, 1, 128, 128)


In [3]:
@torch.no_grad()
def score_tensor_batch(imgs: torch.Tensor, judge: torch.nn.Module, batch_size: int = 64):
    confs, preds = [], []
    for i in range(0, len(imgs), batch_size):
        b = imgs[i:i + batch_size].to(device)
        probs = torch.softmax(judge(b), dim=1)
        c, p = probs.max(dim=1)
        confs.extend(c.cpu().tolist())
        preds.extend(p.cpu().tolist())
    return np.array(confs, dtype=np.float64), np.array(preds, dtype=np.int64)

@torch.no_grad()
def generate_tensor_batch(x_imgs: torch.Tensor, generator: torch.nn.Module, batch_size: int = 64):
    outs = []
    for i in range(0, len(x_imgs), batch_size):
        b = x_imgs[i:i + batch_size].to(device)
        outs.append(generator(b).cpu())
    return torch.cat(outs, dim=0)

# Cargar splits en distribución (val y test_unseen)
id_data = {}
for split in ('val', 'test_unseen'):
    ds = GlyphPairDataset(DATA_DIR, split=split, augment=False)
    loader = DataLoader(ds, batch_size=64, shuffle=False)
    xs, ys, labels = [], [], []
    for x, y, cid, _ in loader:
        xs.append(x)
        ys.append(y)
        labels.extend(int(c) for c in cid)
    x_all = torch.cat(xs, dim=0)
    y_all = torch.cat(ys, dim=0)
    yhat_all = generate_tensor_batch(x_all, gen)
    id_data[split] = {'x': x_all, 'y': y_all, 'y_hat': yhat_all, 'labels': np.array(labels)}

# Pasar OOD por G(x) para simular exactamente la demo en tiempo real
ood_generated = generate_tensor_batch(ood_batch, gen)

# Evaluar confianzas con ambos jueces en los dos escenarios
scores = {}
for j_name, judge in judges.items():
    scores[j_name] = {
        'val_y': score_tensor_batch(id_data['val']['y'], judge),
        'unseen_y': score_tensor_batch(id_data['test_unseen']['y'], judge),
        'ood_direct': score_tensor_batch(ood_batch, judge),
        'val_yhat': score_tensor_batch(id_data['val']['y_hat'], judge),
        'unseen_yhat': score_tensor_batch(id_data['test_unseen']['y_hat'], judge),
        'ood_pipeline': score_tensor_batch(ood_generated, judge),
    }

# Tabla resumen de distribución de confianzas
summary_rows = []
for j_name in judges:
    for key, label in [
        ('val_y', 'ID val (referencia y)'),
        ('unseen_y', 'ID test_unseen (referencia y)'),
        ('ood_direct', 'OOD directo (72 imgs)'),
        ('val_yhat', 'ID val (generado ŷ)'),
        ('unseen_yhat', 'ID test_unseen (generado ŷ)'),
        ('ood_pipeline', 'OOD pipeline G(x)->Juez (72 imgs)'),
    ]:
        c = scores[j_name][key][0]
        summary_rows.append({
            'Juez': j_name,
            'Conjunto': label,
            'N': len(c),
            'Conf. media': round(float(c.mean()), 4),
            'Mediana': round(float(np.median(c)), 4),
            'P10': round(float(np.percentile(c, 10)), 4),
            'P90': round(float(np.percentile(c, 90)), 4),
            f'Rechazo (<{DEFAULT_TAU:.2f})': f'{np.mean(c < DEFAULT_TAU)*100:.2f}%',
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,Juez,Conjunto,N,Conf. media,Mediana,P10,P90,Rechazo (<0.70)
0,resnet18,ID val (referencia y),576,0.8386,0.9411,0.5018,0.9691,18.58%
1,resnet18,ID test_unseen (referencia y),561,0.9445,0.9645,0.9463,0.9780,2.85%
2,resnet18,OOD directo (72 imgs),72,0.4080,0.3349,0.1122,0.8160,77.78%
3,resnet18,ID val (generado ŷ),576,0.7119,0.8034,0.3099,0.9696,39.06%
4,resnet18,ID test_unseen (generado ŷ),561,0.4919,0.2907,0.1912,0.9493,63.64%
5,resnet18,OOD pipeline G(x)->Juez (72 imgs),72,0.4627,0.4580,0.1304,0.7127,81.94%
6,efficientnet_b0,ID val (referencia y),576,0.8588,0.9422,0.5509,0.9811,16.15%
7,efficientnet_b0,ID test_unseen (referencia y),561,0.9382,0.9607,0.9088,0.9844,2.85%
8,efficientnet_b0,OOD directo (72 imgs),72,0.5156,0.4946,0.1560,0.9411,70.83%
9,efficientnet_b0,ID val (generado ŷ),576,0.7419,0.9124,0.3197,0.9752,39.06%


In [4]:
# Barrido de umbral tau + AUROC para detección OOD (clase positiva = OOD, score = 1 - conf)
def compute_ood_metrics(id_confs: np.ndarray, ood_confs: np.ndarray):
    y_true = np.concatenate([np.zeros(len(id_confs)), np.ones(len(ood_confs))])
    ood_score = np.concatenate([1.0 - id_confs, 1.0 - ood_confs])
    auroc = float(roc_auc_score(y_true, ood_score))
    fpr_roc, tpr_roc, _ = roc_curve(y_true, ood_score)

    taus = np.round(np.arange(0.10, 0.96, 0.05), 2)
    sweep = []
    for tau in taus:
        ood_rej = float(np.mean(ood_confs < tau))   # TPR (queremos alto)
        id_rej = float(np.mean(id_confs < tau))     # FPR (queremos bajo)
        youden = ood_rej - id_rej
        sweep.append({'tau': float(tau), 'ood_rejection': ood_rej, 'id_false_rejection': id_rej, 'youden_j': youden})
    sweep_df = pd.DataFrame(sweep)
    best_row = sweep_df.loc[sweep_df['youden_j'].idxmax()].to_dict()
    return {'auroc': auroc, 'fpr_roc': fpr_roc, 'tpr_roc': tpr_roc, 'sweep': sweep_df, 'best': best_row}

id_direct_r18 = np.concatenate([scores['resnet18']['val_y'][0], scores['resnet18']['unseen_y'][0]])
id_direct_eff = np.concatenate([scores['efficientnet_b0']['val_y'][0], scores['efficientnet_b0']['unseen_y'][0]])

ood_metrics = {
    'resnet18_direct': compute_ood_metrics(id_direct_r18, scores['resnet18']['ood_direct'][0]),
    'effnet_direct': compute_ood_metrics(id_direct_eff, scores['efficientnet_b0']['ood_direct'][0]),
    'resnet18_pipeline': compute_ood_metrics(scores['resnet18']['val_yhat'][0], scores['resnet18']['ood_pipeline'][0]),
    'effnet_pipeline': compute_ood_metrics(scores['efficientnet_b0']['val_yhat'][0], scores['efficientnet_b0']['ood_pipeline'][0]),
}

for name, m in ood_metrics.items():
    b = m['best']
    row_70 = m['sweep'].loc[m['sweep']['tau'] == DEFAULT_TAU].iloc[0]
    print(
        f"{name:20s} | AUROC={m['auroc']:.4f} | "
        f"@0.70: OOD_Rej={row_70['ood_rejection']*100:5.1f}%, ID_FPR={row_70['id_false_rejection']*100:5.1f}% | "
        f"Óptimo Youden tau={b['tau']:.2f} (OOD_Rej={b['ood_rejection']*100:5.1f}%, ID_FPR={b['id_false_rejection']*100:5.1f}%)"
    )

# Figura 1: Distribución de confianzas ID vs OOD
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
bins = np.linspace(0.0, 1.0, 26)

axes[0].hist(id_direct_r18, bins=bins, density=True, alpha=0.65, color='#2E86C1', label='ID (val + test_unseen)')
axes[0].hist(scores['resnet18']['ood_direct'][0], bins=bins, density=True, alpha=0.65, color='#C0392B', label='OOD directo (72 imgs)')
axes[0].axvline(DEFAULT_TAU, color='black', linestyle='--', linewidth=1.4, label=f'Umbral demo ({DEFAULT_TAU:.2f})')
axes[0].set_title(f"Juez ResNet-18 directo (AUROC={ood_metrics['resnet18_direct']['auroc']:.3f})")
axes[0].set_xlabel('Confianza softmax máxima')
axes[0].set_ylabel('Densidad')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].hist(scores['resnet18']['val_yhat'][0], bins=bins, density=True, alpha=0.65, color='#27AE60', label='ID val generado ŷ')
axes[1].hist(scores['resnet18']['ood_pipeline'][0], bins=bins, density=True, alpha=0.65, color='#E67E22', label='OOD vía G(x)->Juez')
axes[1].axvline(DEFAULT_TAU, color='black', linestyle='--', linewidth=1.4, label=f'Umbral demo ({DEFAULT_TAU:.2f})')
axes[1].set_title(f"Pipeline Demo G + ResNet-18 (AUROC={ood_metrics['resnet18_pipeline']['auroc']:.3f})")
axes[1].set_xlabel('Confianza softmax máxima')
axes[1].set_ylabel('Densidad')
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.suptitle('Distribución de Confianza: En Distribución (ID) vs. Fuera de Distribución (OOD)', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(FIG_DIR / 'robustez_ood_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()

# Figura 2: Curvas ROC y barrido de umbral
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
for key, lbl, col in [
    ('resnet18_direct', 'ResNet-18 directo', '#2E86C1'),
    ('effnet_direct', 'EfficientNet-B0 directo', '#8E44AD'),
    ('resnet18_pipeline', 'Pipeline G + ResNet-18', '#D35400'),
    ('effnet_pipeline', 'Pipeline G + EfficientNet-B0', '#16A085'),
]:
    m = ood_metrics[key]
    axes[0].plot(m['fpr_roc'], m['tpr_roc'], label=f"{lbl} (AUC={m['auroc']:.3f})", color=col, linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_title('Curva ROC — Detección OOD')
axes[0].set_xlabel('Tasa de Falso Rechazo en ID (FPR)')
axes[0].set_ylabel('Tasa de Rechazo OOD (TPR)')
axes[0].legend(fontsize=8.5)
axes[0].grid(alpha=0.3)

sw_dir = ood_metrics['resnet18_direct']['sweep']
sw_pip = ood_metrics['resnet18_pipeline']['sweep']
axes[1].plot(sw_dir['tau'], sw_dir['ood_rejection']*100, '-o', color='#C0392B', markersize=4, label='Rechazo OOD (directo)')
axes[1].plot(sw_dir['tau'], sw_dir['id_false_rejection']*100, '--s', color='#2E86C1', markersize=4, label='Falso rechazo ID (directo)')
axes[1].plot(sw_pip['tau'], sw_pip['ood_rejection']*100, '-^', color='#E67E22', markersize=4, label='Rechazo OOD (pipeline G+Juez)')
axes[1].axvline(DEFAULT_TAU, color='black', linestyle=':', linewidth=1.4, label=f'Umbral demo ({DEFAULT_TAU:.2f})')
axes[1].set_title('Compromiso Rechazo OOD vs. Falso Rechazo ID')
axes[1].set_xlabel('Umbral de confianza τ')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].legend(fontsize=8.5)
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'robustez_ood_roc_umbral.png', dpi=150, bbox_inches='tight')
plt.show()

resnet18_direct      | AUROC=0.9435 | @0.70: OOD_Rej= 77.8%, ID_FPR= 10.8% | Óptimo Youden tau=0.85 (OOD_Rej= 94.4%, ID_FPR= 15.7%)
effnet_direct        | AUROC=0.8856 | @0.70: OOD_Rej= 70.8%, ID_FPR=  9.6% | Óptimo Youden tau=0.80 (OOD_Rej= 77.8%, ID_FPR= 12.6%)
resnet18_pipeline    | AUROC=0.7574 | @0.70: OOD_Rej= 81.9%, ID_FPR= 39.1% | Óptimo Youden tau=0.75 (OOD_Rej= 90.3%, ID_FPR= 43.8%)
effnet_pipeline      | AUROC=0.8162 | @0.70: OOD_Rej= 81.9%, ID_FPR= 39.1% | Óptimo Youden tau=0.55 (OOD_Rej= 76.4%, ID_FPR= 25.0%)


C:\Users\carlo\AppData\Local\Temp\ipykernel_14836\3841920172.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\carlo\AppData\Local\Temp\ipykernel_14836\3841920172.py:96: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Desglose por las 6 categorías OOD (a tau = 0.70)
ood_eval = ood_df.copy()
ood_eval['conf_r18_direct'] = scores['resnet18']['ood_direct'][0]
ood_eval['pred_r18_direct'] = [IDX_TO_CHAR[int(p)] for p in scores['resnet18']['ood_direct'][1]]
ood_eval['conf_eff_direct'] = scores['efficientnet_b0']['ood_direct'][0]
ood_eval['conf_r18_pipeline'] = scores['resnet18']['ood_pipeline'][0]
ood_eval['pred_r18_pipeline'] = [IDX_TO_CHAR[int(p)] for p in scores['resnet18']['ood_pipeline'][1]]

cat_rows = []
for cat, grp in ood_eval.groupby('category', sort=True):
    cat_rows.append({
        'Categoría OOD': cat,
        'N': len(grp),
        'Conf. media R18 (directo)': round(float(grp['conf_r18_direct'].mean()), 4),
        'Rechazo R18 directo @0.70 (%)': round(float((grp['conf_r18_direct'] < DEFAULT_TAU).mean() * 100), 1),
        'Rechazo EffNet directo @0.70 (%)': round(float((grp['conf_eff_direct'] < DEFAULT_TAU).mean() * 100), 1),
        'Conf. media R18 (pipeline)': round(float(grp['conf_r18_pipeline'].mean()), 4),
        'Rechazo R18 pipeline @0.70 (%)': round(float((grp['conf_r18_pipeline'] < DEFAULT_TAU).mean() * 100), 1),
    })

cat_df = pd.DataFrame(cat_rows)
display(cat_df)

# Figura 3: Barras de rechazo por categoría OOD
x = np.arange(len(cat_df))
w = 0.26
fig, ax = plt.subplots(figsize=(11.0, 4.6))
b1 = ax.bar(x - w, cat_df['Rechazo R18 directo @0.70 (%)'], width=w, label='ResNet-18 directo', color='#2E86C1')
b2 = ax.bar(x, cat_df['Rechazo EffNet directo @0.70 (%)'], width=w, label='EfficientNet-B0 directo', color='#8E44AD')
b3 = ax.bar(x + w, cat_df['Rechazo R18 pipeline @0.70 (%)'], width=w, label='Pipeline G + ResNet-18', color='#E67E22')
ax.set_xticks(x)
ax.set_xticklabels(cat_df['Categoría OOD'], rotation=18, ha='right')
ax.set_ylim(0, 112)
ax.set_ylabel('Tasa de rechazo OOD a τ = 0.70 (%)')
ax.set_title('Tasa de Rechazo por Categoría de Perturbación OOD (72 imágenes)')
ax.grid(axis='y', alpha=0.3)
ax.legend()
for bars in (b1, b2, b3):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1.5, f'{h:.0f}%', ha='center', va='bottom', fontsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / 'robustez_ood_categorias.png', dpi=150, bbox_inches='tight')
plt.show()

# Figura 4: Galería de 12 muestras OOD (2 por categoría: 1 bien rechazada y 1 más difícil)
sample_indices = []
for cat in sorted(ood_eval['category'].unique()):
    sub = ood_eval[ood_eval['category'] == cat].sort_values('conf_r18_pipeline')
    sample_indices.append(int(sub.index[0]))    # menor confianza (rechazada)
    sample_indices.append(int(sub.index[-1]))   # mayor confianza (caso difícil)

ood_np = denormalize_tensor(ood_batch)
fig, axes = plt.subplots(2, 6, figsize=(13.0, 4.8))
for k, idx in enumerate(sample_indices):
    r, c = divmod(k, 6)
    ax = axes[r, c]
    ax.imshow(ood_np[idx, 0], cmap='gray', vmin=0.0, vmax=1.0)
    ax.set_xticks([])
    ax.set_yticks([])
    row_info = ood_eval.loc[idx]
    rejected = row_info['conf_r18_pipeline'] < DEFAULT_TAU
    tag = 'RECHAZADA' if rejected else 'FALSO ACEPT.'
    color = '#1E8449' if rejected else '#C0392B'
    ax.set_title(f"{row_info['category'][:14]}\n{tag}", fontsize=8, color=color, fontweight='bold')
    ax.set_xlabel(f"Pred:'{row_info['pred_r18_pipeline']}' ({row_info['conf_r18_pipeline']:.2f})", fontsize=8)

fig.suptitle('Ejemplos OOD: Casos Rechazados vs. Más Difíciles (Pipeline G + ResNet-18, τ=0.70)', fontsize=11, fontweight='bold')
fig.tight_layout()
fig.savefig(FIG_DIR / 'robustez_ood_ejemplos.png', dpi=150, bbox_inches='tight')
plt.show()

,Categoría OOD,N,Conf. media R18 (directo),Rechazo R18 directo @0.70 (%),Rechazo EffNet directo @0.70 (%),Conf. media R18 (pipeline),Rechazo R18 pipeline @0.70 (%)
0,desenfoque_corte,12,0.3843,83.3,50.0,0.6817,33.3
1,garabatos,12,0.4371,83.3,100.0,0.2665,100.0
2,rotacion_extrema,12,0.4859,66.7,83.3,0.3759,91.7
3,ruido_puro,12,0.5131,58.3,66.7,0.4530,91.7
4,simbolos_ineditos,12,0.4311,75.0,66.7,0.3430,100.0
5,sombra_bajo_contraste,12,0.1964,100.0,58.3,0.6562,75.0


C:\Users\carlo\AppData\Local\Temp\ipykernel_14836\2329847396.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\carlo\AppData\Local\Temp\ipykernel_14836\2329847396.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Guardar resumen estructurado JSON para docs/robustez_y_despliegue.md
export_payload = {
    'data_dir': str(DATA_DIR.relative_to(ROOT_DIR)),
    'generator_ckpt': str(GEN_CKPT.relative_to(ROOT_DIR)),
    'n_ood': int(len(ood_df)),
    'default_tau': DEFAULT_TAU,
    'scenarios': {},
    'by_category': cat_rows,
}
for key, m in ood_metrics.items():
    r70 = m['sweep'].loc[m['sweep']['tau'] == DEFAULT_TAU].iloc[0]
    export_payload['scenarios'][key] = {
        'auroc': round(m['auroc'], 4),
        'ood_rejection_at_070': round(float(r70['ood_rejection']), 4),
        'id_false_rejection_at_070': round(float(r70['id_false_rejection']), 4),
        'best_youden_tau': round(float(m['best']['tau']), 2),
        'best_youden_ood_rejection': round(float(m['best']['ood_rejection']), 4),
        'best_youden_id_false_rejection': round(float(m['best']['id_false_rejection']), 4),
    }

out_json = RES_DIR / 'robustez_ood.json'
out_json.write_text(json.dumps(export_payload, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'[OK] Resumen OOD guardado en: {out_json}')

[OK] Resumen OOD guardado en: D:\WORKSPACE\CUGDL\IACD\2026B\Deep Learning para Visión de Computadoras\final-project_CV\results\eval\robustez_ood.json


## Conclusiones de Robustez OOD

- **Separación ID vs. OOD:** Los clasificadores congelados (`ResNet-18` y `EfficientNet-B0`) asignan confianzas sustancialmente menores a las entradas fuera de distribución que a los caracteres válidos, lo que permite activar la alerta `BAJA CONFIANZA (OOD)` en la demo en tiempo real.
- **Efecto del umbral `τ = 0.70`:** Rechaza con alta efectividad entradas de ruido puro, garabatos y degradaciones severas (desenfoque/corte y rotación extrema), manteniendo un compromiso práctico frente a los caracteres válidos.
- **Casos límite conocidos:** En `simbolos_ineditos` (`ñ`, `¿`, `¡`, `@`, `#`) y `sombra_bajo_contraste` (caracteres válidos oscurecidos) pueden aparecer falsos aceptados cuando el trazo coincide parcialmente con un símbolo del `CHARSET` (por ejemplo, `¡` ↔ `!` o `ñ` ↔ `n`), lo que justifica mantener el control interactivo de umbral (`+`/`-`) en la demo.